# OMSL Coffee Multi-Source — Colab Resumable

Pilot validation Coffee17 + CBD untuk membandingkan **OMSL0** dan **OMSL-TC**. Checkpoint langsung disimpan ke Google Drive. Sebelum mulai, aktifkan GPU dan buat Colab Secret bernama `ROBOFLOW_API_KEY`.

In [ ]:
# 1/6 — Setup repository, Drive, dan dependency
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, subprocess, sys

REPO = Path('/content/coffee-bean-classification')
REPO_URL = 'https://github.com/ediprin/coffee-bean-classification.git'
REPO_REF = 'main'
DRIVE_ROOT = Path('/content/drive/MyDrive/coffee-bean-classification/omsl')
DATA_ROOT = Path('/content/omsl-data')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_REF], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO), 'roboflow', 'kagglehub'], check=True)
os.chdir(REPO)
print('SETUP SIAP:', REPO)
print('OUTPUT    :', DRIVE_ROOT)

In [ ]:
# 2/6 — Unduh dan audit Coffee17 (aman diulang setelah proses terputus)
import shutil
import kagglehub

coffee_original = DATA_ROOT / 'coffee17-original'
coffee_clean = DATA_ROOT / 'coffee17-clean'

coffee_source = coffee_original / 'source'
prepared_images = list(coffee_source.glob('*/*/*')) if coffee_source.is_dir() else []
if coffee_source.exists() and len(prepared_images) != 979:
    print(f'Membersihkan preparasi Coffee17 parsial ({len(prepared_images)}/979)...')
    shutil.rmtree(coffee_original)

if not coffee_source.is_dir():
    coffee_raw = Path(kagglehub.dataset_download(
        'sujitraarw/coffee-green-bean-with-17-defects-original'
    ))
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_coffee17',
        '--output', str(coffee_original), '--raw-root', str(coffee_raw), '--seed', '42'
    ], check=True)
else:
    print('SKIP: Coffee17 original sudah lengkap.')

if coffee_clean.exists() and not (coffee_clean / 'audit.json').is_file():
    print('Membersihkan grouped-fold Coffee17 parsial...')
    shutil.rmtree(coffee_clean)
if not (coffee_clean / 'audit.json').is_file():
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_clean_grouped_folds',
        '--source-root', str(coffee_original / 'source'),
        '--output-root', str(coffee_clean),
        '--expected-count', '979', '--folds', '5', '--seed', '42',
        '--validation-ratio', '0.1'
    ], check=True)

COFFEE_ROOT = coffee_clean / 'folds/fold_1/source'
assert (COFFEE_ROOT / 'train').is_dir() and (COFFEE_ROOT / 'val').is_dir()
print('COFFEE17 SIAP:', COFFEE_ROOT)

In [ ]:
# 3/6 — Unduh Roboflow CBD memakai Colab Secret
from google.colab import userdata
from roboflow import Roboflow

api_key = userdata.get('ROBOFLOW_API_KEY')
assert api_key, 'Secret ROBOFLOW_API_KEY belum tersedia.'
cbd_raw = DATA_ROOT / 'cbd-raw'
cbd_prepared = DATA_ROOT / 'cbd-prepared'

if not any(cbd_raw.rglob('*.jpg')):
    rf = Roboflow(api_key=api_key)
    version = rf.workspace('situju-kamkape').project('cbd-multiclassify-bdmnb').version(1)
    downloaded = version.download('folder', location=str(cbd_raw))
    cbd_raw = Path(downloaded.location)

if not (cbd_prepared / 'audit.json').is_file():
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.data.preparation.prepare_cbd_multiclassify',
        '--raw-root', str(cbd_raw), '--output-root', str(cbd_prepared), '--seed', '42'
    ], check=True)

CBD_ROOT = cbd_prepared / 'source'
assert (CBD_ROOT / 'train').is_dir() and (CBD_ROOT / 'val').is_dir()
print('CBD SIAP:', CBD_ROOT)

In [ ]:
# 4/6 — Buat config runtime dengan path aktual
import copy, json, yaml

ONTOLOGY = REPO / 'configs/omsl/coffee_ontology_draft.yaml'
runtime_configs = {}
for code, template_name in {
    'OMSL0': 'OMSL0_efficientnetv2_gap.yaml',
    'OMSL1': 'OMSL1_efficientnetv2_gap_tc.yaml',
}.items():
    cfg = yaml.safe_load((REPO / 'configs/omsl' / template_name).read_text())
    cfg['ontology']['path'] = str(ONTOLOGY)
    cfg['data']['sources'][0]['root'] = str(COFFEE_ROOT)
    cfg['data']['sources'][1]['root'] = str(CBD_ROOT)
    cfg['training']['output_dir'] = str(DRIVE_ROOT / f'{code}_seed123')
    cfg['seed'] = 123
    path = Path('/content') / f'{code}_runtime.yaml'
    path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    runtime_configs[code] = path
    print(code, '->', path, '| output:', cfg['training']['output_dir'])

In [ ]:
# 5/6 — Pilot seed 123; progress tampil per epoch dan aman untuk resume
for code in ('OMSL0', 'OMSL1'):
    print(f'\n=== MULAI / RESUME {code} ===', flush=True)
    subprocess.run([
        sys.executable, '-u', '-m', 'bilinear_lmmd.engine.train_omsl',
        '--config', str(runtime_configs[code]), '--seed', '123', '--resume'
    ], check=True)
print('PILOT SELESAI')

In [ ]:
# 6/6 — Ringkasan validation tanpa membuka test
reports = {}
for code in ('OMSL0', 'OMSL1'):
    path = DRIVE_ROOT / f'{code}_seed123/metrics.json'
    assert path.is_file(), f'Report belum ada: {path}'
    reports[code] = json.loads(path.read_text())
    row = reports[code]
    print(f"{code}: dataset-Macro={row['dataset_macro_f1']:.2%} balanced={row['dataset_balanced_accuracy']:.2%}")
    for name, metrics in row['datasets'].items():
        print(f"  {name:10s} Macro={metrics['observed_macro_f1']:.2%} Acc={metrics['observed_accuracy']:.2%}")
delta = reports['OMSL1']['dataset_macro_f1'] - reports['OMSL0']['dataset_macro_f1']
print(f'\nDelta OMSL-TC vs OMSL: {delta:+.2%}')